# Music Annotation Project — Notebook 36
## V8.2g Blinded Utility-versus-`p_print` Group Review

**Runtime:** CPU is sufficient.

This is a development-only, read-only human-review experiment. It does **not** retrain, refit, recalibrate, alter V8.2 probabilities, change Notebook 35 group definitions, or authorise production use.

It asks one narrow question:

> Among the top 50 near-collinear groups ranked by reconstruction-avoided utility and the top 50 ranked by mean saved `p_print`, which selection method yields the higher human-reviewed beneficial rate?

The two top-50 lists are united and de-duplicated before review. In the current lineage this is expected to produce 95 unique groups, but the notebook computes and records the actual number rather than hard-coding it.

### Blinding contract

During review, the interface hides:

- ranking origin;
- utility and `p_print` values;
- clean diagnostics;
- difference maps;
- whether Image A or Image B is the V7.8 baseline.

Each record shows only two identically cropped images, labelled **Image A** and **Image B**, with the same hollow purple group box. One is V7.8; the other is V7.8 plus exact observed-byte preservation at the declared group coordinates. Left/right assignment and review order are deterministic but blinded.

### Review choices

- **Image A visibly better**
- **Image B visibly better**
- **No visible difference**
- **Mixed / uncertain effect**

After all union groups are reviewed, the notebook decodes the blinded side assignment and produces:

- beneficial, harmful, no-difference and uncertain rates for each ranking;
- Wilson 95% intervals for the group-level beneficial rates;
- page-clustered bootstrap 95% intervals for each beneficial rate and their difference;
- a rank-position paired comparison across the two top-50 lists;
- CSV, Excel and JSON records.

The review result is diagnostic development evidence, not a new automatic policy.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "gradio>=5.0" opencv-python-headless pandas numpy pyarrow openpyxl scipy

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, os, platform, sys, threading, re, ast as pyast

import cv2
import numpy as np
import pandas as pd
import gradio as gr
from scipy.stats import binomtest

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# Notebook 35 source. Set this when resuming against a particular audit.
SOURCE_V82F_RUN_OVERRIDE = None

# Set this to an existing Notebook 36 run to resume completed reviews.
REVIEW_RUN_OVERRIDE = None

V82F_ROOT_CANDIDATES = [
    ROOT / 'observed_print_classifier_v8_2f_reconstruction_avoided',
    ROOT / 'observed_print_classifier_v8_2f_reconstruction_avoided_counterfactual_audit',
    ROOT / 'observed_print_classifier_v8_2f_counterfactual_audit',
]
OUTPUT_ROOT = ROOT / 'observed_print_classifier_v8_2g_blinded_group_review'

TOP_K = 50
REVIEW_SEED = 36082
BOOTSTRAP_SEED = 36083
BOOTSTRAP_RESAMPLES = 10000
CONFIDENCE_LEVEL = 0.95
CROP_MARGIN = 42
MIN_CROP_HALF_SIZE = 52
DISPLAY_SCALE = 3
BOX_MARGIN = 4
PUBLIC_LINK = True
REVIEWER_NAME = 'David Bolton'

DECISIONS = [
    'Image A visibly better',
    'Image B visibly better',
    'No visible difference',
    'Mixed / uncertain effect',
]

if REVIEW_RUN_OVERRIDE is None:
    RUN = datetime.now().strftime('%Y%m%d_%H%M%S') + '_v8_2g_blinded_group_review'
    OUT = OUTPUT_ROOT / RUN
    OUT.mkdir(parents=True, exist_ok=False)
else:
    OUT = Path(REVIEW_RUN_OVERRIDE)
    if not OUT.exists():
        raise RuntimeError(f'Review run override does not exist: {OUT}')

TABLES = OUT / 'tables'
REVIEW = OUT / 'review'
AUDIT = OUT / 'audit'
EXPORTS = OUT / 'exports'
for p in [TABLES, REVIEW, AUDIT, EXPORTS]:
    p.mkdir(parents=True, exist_ok=True)

print('Notebook 36 output:', OUT)

## 2. Resolve the latest complete Notebook 35 run

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()


def complete_runs(root, required):
    hits = []
    root = Path(root)
    if root.exists():
        for p in root.iterdir():
            if p.is_dir() and all((p / r).exists() for r in required):
                hits.append(p)
    return sorted(hits, key=lambda p: p.stat().st_mtime)


REQUIRED_35 = [
    'experiment_identity.json',
    'tables/near_collinear_group_counterfactuals.csv',
]

if SOURCE_V82F_RUN_OVERRIDE is not None:
    V82F = Path(SOURCE_V82F_RUN_OVERRIDE)
    missing = [r for r in REQUIRED_35 if not (V82F / r).exists()]
    if missing:
        raise RuntimeError(f'Incomplete Notebook 35 source override: {missing}')
else:
    hits = []
    for root in V82F_ROOT_CANDIDATES:
        hits.extend(complete_runs(root, REQUIRED_35))
    hits = sorted(set(hits), key=lambda p: p.stat().st_mtime)
    if not hits:
        raise RuntimeError(
            'No complete Notebook 35 run was found. Set SOURCE_V82F_RUN_OVERRIDE '
            'to the run containing experiment_identity.json and '
            'tables/near_collinear_group_counterfactuals.csv.'
        )
    V82F = hits[-1]

ID35 = json.loads((V82F / 'experiment_identity.json').read_text())
GROUP_PATH = V82F / 'tables' / 'near_collinear_group_counterfactuals.csv'
groups = pd.read_csv(GROUP_PATH)
if groups.empty:
    raise RuntimeError('Notebook 35 group table is empty.')

print('Notebook 35 source:', V82F)
print('Groups:', len(groups))
print('Columns:', groups.columns.tolist())

## 3. Resolve ranking columns, group identity and source pages

The code deliberately recognises several plausible column names, because the Notebook 35 table may use either concise or descriptive field names. It fails with a clear list of available columns if the required fields cannot be found.

In [ ]:
def first_existing(columns, candidates, role):
    for c in candidates:
        if c in columns:
            return c
    raise RuntimeError(f'Cannot resolve {role}. Tried {candidates}. Available: {list(columns)}')


GROUP_ID_COL = first_existing(
    groups.columns,
    ['group_id', 'near_collinear_group_id', 'group_uid', 'group_key', 'id'],
    'group ID column',
)
PAGE_COL = first_existing(groups.columns, ['page', 'page_id'], 'page column')
UTILITY_COL = first_existing(
    groups.columns,
    ['group_utility', 'diagnostic_group_utility', 'declared_diagnostic_group_utility', 'utility'],
    'group utility column',
)
PPRINT_COL = first_existing(
    groups.columns,
    ['mean_p_print', 'mean_saved_p_print', 'p_print_mean', 'group_mean_p_print'],
    'mean p_print column',
)

for c in [UTILITY_COL, PPRINT_COL]:
    groups[c] = pd.to_numeric(groups[c], errors='raise').astype(float)
groups[PAGE_COL] = groups[PAGE_COL].astype(str)
groups[GROUP_ID_COL] = groups[GROUP_ID_COL].astype(str)
if groups[GROUP_ID_COL].duplicated().any():
    raise RuntimeError(f'Duplicate group IDs in {GROUP_PATH}')


def recursive_find_paths(obj, key_fragments):
    found = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            kl = str(k).lower()
            if isinstance(v, str) and any(f in kl for f in key_fragments):
                found.append(v)
            found.extend(recursive_find_paths(v, key_fragments))
    elif isinstance(obj, list):
        for v in obj:
            found.extend(recursive_find_paths(v, key_fragments))
    return found


v82_candidates = recursive_find_paths(ID35, ['source_v82_run', 'source_v8_2_run', 'v82_run'])
V82 = next((Path(x) for x in v82_candidates if Path(x).exists()), None)
if V82 is None:
    raise RuntimeError(
        'Notebook 35 identity did not resolve an existing V8.2 source run. '
        'The review needs the source whole_pages/<page>/v7_8.png and observed.png files.'
    )

print('Group ID:', GROUP_ID_COL)
print('Utility:', UTILITY_COL)
print('Mean p_print:', PPRINT_COL)
print('V8.2 source:', V82)

## 4. Resolve exact group-member coordinates

In [ ]:
def normalise_xy_frame(df, id_col):
    cols = df.columns
    page_col = next((c for c in ['page', 'page_id'] if c in cols), None)
    x_col = next((c for c in ['x', 'pixel_x', 'member_x'] if c in cols), None)
    y_col = next((c for c in ['y', 'pixel_y', 'member_y'] if c in cols), None)
    if page_col is None or x_col is None or y_col is None or id_col not in cols:
        return None
    out = df[[id_col, page_col, y_col, x_col]].copy()
    out.columns = ['group_id', 'page', 'y', 'x']
    out['group_id'] = out.group_id.astype(str)
    out['page'] = out.page.astype(str)
    out['y'] = pd.to_numeric(out.y, errors='raise').astype(int)
    out['x'] = pd.to_numeric(out.x, errors='raise').astype(int)
    return out.drop_duplicates(['group_id', 'page', 'y', 'x'])


def try_membership_tables():
    preferred = [
        'near_collinear_group_members.parquet',
        'near_collinear_group_members.csv',
        'group_membership.parquet',
        'group_membership.csv',
        'near_collinear_candidate_pixels.parquet',
        'near_collinear_candidate_pixels.csv',
        'pixel_counterfactual_audit.parquet',
        'pixel_counterfactual_audit.csv',
    ]
    candidates = []
    for name in preferred:
        p = V82F / 'tables' / name
        if p.exists():
            candidates.append(p)
    for p in sorted((V82F / 'tables').glob('*')):
        if p.suffix.lower() in {'.csv', '.parquet'} and p not in candidates:
            if any(t in p.name.lower() for t in ['group', 'member', 'pixel']):
                candidates.append(p)

    for p in candidates:
        try:
            df = pd.read_parquet(p) if p.suffix.lower() == '.parquet' else pd.read_csv(p)
        except Exception:
            continue
        id_candidates = [GROUP_ID_COL, 'group_id', 'near_collinear_group_id', 'group_uid', 'group_key']
        for id_col in id_candidates:
            q = normalise_xy_frame(df, id_col)
            if q is not None and q.group_id.isin(groups[GROUP_ID_COL].astype(str)).any():
                print('Resolved group members from:', p)
                return q, p
    return None, None


def parse_coordinate_value(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return value
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    for parser in [json.loads, pyast.literal_eval]:
        try:
            return parser(text)
        except Exception:
            pass
    return None


def membership_from_serialised_groups():
    rows = []

    # Notebook 35 stores coordinates as:
    # y:x|y:x|y:x
    if 'coordinates' in groups.columns:
        for _, r in groups.iterrows():
            value = r['coordinates']

            if pd.isna(value):
                continue

            text = str(value).strip()
            if not text:
                continue

            for token in text.split('|'):
                token = token.strip()
                if not token:
                    continue

                parts = token.split(':')
                if len(parts) != 2:
                    raise RuntimeError(
                        f"Malformed coordinate token {token!r} "
                        f"for group {r[GROUP_ID_COL]}"
                    )

                # Notebook 35 writes coordinates as y:x.
                y, x = map(int, parts)

                rows.append({
                    'group_id': str(r[GROUP_ID_COL]),
                    'page': str(r[PAGE_COL]),
                    'y': y,
                    'x': x,
                })

        if rows:
            return (
                pd.DataFrame(rows)
                .drop_duplicates(['group_id', 'page', 'y', 'x'])
                .reset_index(drop=True)
            )

    return None


members, MEMBERSHIP_SOURCE = try_membership_tables()
if members is None:
    members = membership_from_serialised_groups()
    MEMBERSHIP_SOURCE = GROUP_PATH if members is not None else None

if members is None or members.empty:
    raise RuntimeError(
        'Could not resolve exact member x/y coordinates for Notebook 35 groups. '
        'Expected a group-membership/pixel table, or a serialised coordinate column '
        f'in {GROUP_PATH}. Available group columns: {groups.columns.tolist()}'
    )

expected_ids = set(groups[GROUP_ID_COL].astype(str))
resolved_ids = set(members.group_id.astype(str))
missing_ids = expected_ids - resolved_ids
if missing_ids:
    raise RuntimeError(f'Missing exact member coordinates for {len(missing_ids)} groups; examples={sorted(missing_ids)[:10]}')

member_counts = members.groupby('group_id').size()
if 'pixels' in groups.columns:
    declared = groups.set_index(GROUP_ID_COL)['pixels'].astype(int)
    common = declared.index.intersection(member_counts.index)
    mismatch = declared.loc[common] != member_counts.loc[common]
    if mismatch.any():
        bad = declared.loc[common][mismatch].head(10).to_dict()
        raise RuntimeError(f'Group member-count mismatch against Notebook 35 table: {bad}')

print('Resolved member rows:', len(members))
print('Membership source:', MEMBERSHIP_SOURCE)

## 5. Construct and freeze the blinded 95-group union

In [ ]:
utility_top = groups.sort_values([UTILITY_COL, GROUP_ID_COL], ascending=[False, True]).head(TOP_K).copy()
pprint_top = groups.sort_values([PPRINT_COL, GROUP_ID_COL], ascending=[False, True]).head(TOP_K).copy()
if len(utility_top) < TOP_K or len(pprint_top) < TOP_K:
    raise RuntimeError(f'Notebook 35 has fewer than {TOP_K} groups for one or both rankings.')

utility_ids = utility_top[GROUP_ID_COL].astype(str).tolist()
pprint_ids = pprint_top[GROUP_ID_COL].astype(str).tolist()
utility_rank = {gid: i + 1 for i, gid in enumerate(utility_ids)}
pprint_rank = {gid: i + 1 for i, gid in enumerate(pprint_ids)}
union_ids = sorted(set(utility_ids) | set(pprint_ids))

union = groups[groups[GROUP_ID_COL].astype(str).isin(union_ids)].copy()
union['group_id'] = union[GROUP_ID_COL].astype(str)
union['in_utility_top50'] = union.group_id.isin(utility_ids)
union['in_pprint_top50'] = union.group_id.isin(pprint_ids)
union['utility_rank'] = union.group_id.map(utility_rank)
union['pprint_rank'] = union.group_id.map(pprint_rank)

rng = np.random.default_rng(REVIEW_SEED)
shuffled_ids = list(union.group_id)
rng.shuffle(shuffled_ids)
review_order = {gid: i + 1 for i, gid in enumerate(shuffled_ids)}
side_rng = np.random.default_rng(REVIEW_SEED + 1)
forced_side = {gid: ('A' if side_rng.integers(0, 2) == 0 else 'B') for gid in union.group_id}
union['review_order'] = union.group_id.map(review_order).astype(int)
union['forced_side'] = union.group_id.map(forced_side)
union['review_id'] = union.review_order.map(lambda x: f'R{x:03d}')
union = union.sort_values('review_order').reset_index(drop=True)

# Full private manifest contains ranking and side assignment. A public review manifest omits them.
private_manifest = union.copy()
private_manifest.to_csv(AUDIT / 'private_blinding_manifest.csv', index=False)
public_manifest = union[['review_id', 'review_order', 'group_id', PAGE_COL]].copy()
public_manifest.columns = ['review_id', 'review_order', 'group_id', 'page']
public_manifest.to_csv(TABLES / 'blinded_union_sampling_manifest.csv', index=False)

selection_summary = {
    'top_k': TOP_K,
    'utility_top_count': len(utility_ids),
    'pprint_top_count': len(pprint_ids),
    'overlap_count': len(set(utility_ids) & set(pprint_ids)),
    'union_count': len(union_ids),
    'review_seed': REVIEW_SEED,
}
(OUT / 'selection_summary.json').write_text(json.dumps(selection_summary, indent=2))
print(json.dumps(selection_summary, indent=2))

## 6. Render blinded A/B crops

The same hollow purple box is drawn on both images. No clean page, difference map, ranking source, utility value, probability value or baseline/forced label is displayed during review.

In [ ]:
MAGENTA = (255, 0, 255)  # OpenCV BGR


def source_page_paths(page):
    d = V82 / 'whole_pages' / str(page)
    baseline = d / 'v7_8.png'
    observed = d / 'observed.png'
    if not baseline.exists() or not observed.exists():
        raise RuntimeError(f'{page}: missing V8.2 whole-page v7_8.png or observed.png')
    return baseline, observed


def crop_bounds(shape, xs, ys):
    h, w = shape[:2]
    x0, x1 = int(np.min(xs)), int(np.max(xs))
    y0, y1 = int(np.min(ys)), int(np.max(ys))
    cx = (x0 + x1) // 2; cy = (y0 + y1) // 2
    half_w = max(MIN_CROP_HALF_SIZE, (x1 - x0 + 1) // 2 + CROP_MARGIN)
    half_h = max(MIN_CROP_HALF_SIZE, (y1 - y0 + 1) // 2 + CROP_MARGIN)
    bx0 = max(0, cx - half_w); bx1 = min(w, cx + half_w + 1)
    by0 = max(0, cy - half_h); by1 = min(h, cy + half_h + 1)
    return bx0, by0, bx1, by1


def boxed_crop(gray, bx0, by0, bx1, by1, group_xs, group_ys):
    q = cv2.cvtColor(gray[by0:by1, bx0:bx1], cv2.COLOR_GRAY2BGR)
    gx0 = max(0, int(np.min(group_xs) - bx0 - BOX_MARGIN))
    gy0 = max(0, int(np.min(group_ys) - by0 - BOX_MARGIN))
    gx1 = min(q.shape[1] - 1, int(np.max(group_xs) - bx0 + BOX_MARGIN))
    gy1 = min(q.shape[0] - 1, int(np.max(group_ys) - by0 + BOX_MARGIN))
    cv2.rectangle(q, (gx0, gy0), (gx1, gy1), MAGENTA, 2)
    q = cv2.resize(q, (q.shape[1] * DISPLAY_SCALE, q.shape[0] * DISPLAY_SCALE), interpolation=cv2.INTER_NEAREST)
    return q


def render_group_pair(group_id):
    r = private_manifest.loc[private_manifest.group_id.eq(str(group_id))].iloc[0]
    page = str(r[PAGE_COL])
    m = members[members.group_id.eq(str(group_id)) & members.page.eq(page)].copy()
    if m.empty:
        raise RuntimeError(f'{group_id}: no exact members for page {page}')
    baseline_path, observed_path = source_page_paths(page)
    baseline = cv2.imread(str(baseline_path), cv2.IMREAD_GRAYSCALE)
    observed = cv2.imread(str(observed_path), cv2.IMREAD_GRAYSCALE)
    if baseline is None or observed is None or baseline.shape != observed.shape:
        raise RuntimeError(f'{page}: unreadable or mismatched baseline/observed pages')
    ys = m.y.to_numpy(int); xs = m.x.to_numpy(int)
    if (ys < 0).any() or (xs < 0).any() or (ys >= baseline.shape[0]).any() or (xs >= baseline.shape[1]).any():
        raise RuntimeError(f'{group_id}: member coordinate outside page')
    forced = baseline.copy(); forced[ys, xs] = observed[ys, xs]
    # Notebook 35 counterfactual should alter only member coordinates and copy exact observed bytes.
    changed = forced != baseline
    if np.any(changed & ~np.isin(np.arange(baseline.size).reshape(baseline.shape), np.ravel_multi_index((ys, xs), baseline.shape))):
        raise RuntimeError(f'{group_id}: unexpected counterfactual change outside member coordinates')
    bx0, by0, bx1, by1 = crop_bounds(baseline.shape, xs, ys)
    base_crop = boxed_crop(baseline, bx0, by0, bx1, by1, xs, ys)
    forced_crop = boxed_crop(forced, bx0, by0, bx1, by1, xs, ys)
    if r.forced_side == 'A':
        a, b = forced_crop, base_crop
    else:
        a, b = base_crop, forced_crop
    return cv2.cvtColor(a, cv2.COLOR_BGR2RGB), cv2.cvtColor(b, cv2.COLOR_BGR2RGB)

# Preflight every union record before the reviewer is launched.
preflight_rows = []
for _, r in private_manifest.iterrows():
    a, b = render_group_pair(r.group_id)
    preflight_rows.append({
        'review_id': r.review_id,
        'group_id': r.group_id,
        'page': str(r[PAGE_COL]),
        'image_a_shape': str(a.shape),
        'image_b_shape': str(b.shape),
    })
pd.DataFrame(preflight_rows).to_csv(AUDIT / 'render_preflight.csv', index=False)
print('Blinded render preflight passed for', len(preflight_rows), 'groups')

## 7. Review persistence and statistical analysis

In [ ]:
REVIEW_CSV = REVIEW / 'blinded_group_review.csv'
REVIEW_XLSX = REVIEW / 'blinded_group_review.xlsx'
SUMMARY_CSV = TABLES / 'ranking_beneficial_rate_summary.csv'
SUMMARY_XLSX = TABLES / 'ranking_beneficial_rate_summary.xlsx'
PAIRED_CSV = TABLES / 'paired_rank_position_comparison.csv'
BOOTSTRAP_CSV = TABLES / 'page_cluster_bootstrap_rate_difference.csv'
FINAL_JSON = OUT / 'final_review_analysis.json'
LOCK = threading.RLock()

review_df = union[['review_id', 'review_order', 'group_id', PAGE_COL]].copy()
review_df.columns = ['review_id', 'review_order', 'group_id', 'page']
review_df['blind_decision'] = ''
review_df['review_note'] = ''
review_df['reviewer'] = ''
review_df['reviewed_utc'] = ''
review_df['review_revision'] = 0

if REVIEW_CSV.exists():
    old = pd.read_csv(REVIEW_CSV).fillna('')
    required_old = {'review_id', 'group_id', 'blind_decision', 'review_note', 'reviewer', 'reviewed_utc', 'review_revision'}
    if required_old - set(old.columns):
        raise RuntimeError(f'Existing review file is incompatible: {sorted(required_old - set(old.columns))}')
    old['group_id'] = old.group_id.astype(str)
    if set(old.group_id) != set(review_df.group_id):
        raise RuntimeError('Existing review run uses a different blinded union.')
    keep = ['group_id', 'blind_decision', 'review_note', 'reviewer', 'reviewed_utc', 'review_revision']
    review_df = review_df.drop(columns=keep[1:]).merge(old[keep], on='group_id', how='left', validate='one_to_one')
    for c in ['blind_decision', 'review_note', 'reviewer', 'reviewed_utc']:
        review_df[c] = review_df[c].fillna('')
    review_df['review_revision'] = pd.to_numeric(review_df.review_revision, errors='coerce').fillna(0).astype(int)


def persist_review():
    tmp = REVIEW_CSV.with_suffix('.csv.tmp')
    review_df.to_csv(tmp, index=False)
    os.replace(tmp, REVIEW_CSV)
    tmp_x = REVIEW_XLSX.with_name(REVIEW_XLSX.stem + '.tmp.xlsx')
    review_df.to_excel(tmp_x, index=False, engine='openpyxl')
    os.replace(tmp_x, REVIEW_XLSX)


def wilson_interval(successes, n, z=1.959963984540054):
    if n == 0:
        return (np.nan, np.nan)
    p = successes / n
    den = 1 + z*z/n
    centre = p + z*z/(2*n)
    radius = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return ((centre-radius)/den, (centre+radius)/den)


def decoded_reviews():
    q = private_manifest.merge(
        review_df[['group_id', 'blind_decision', 'review_note', 'reviewer', 'reviewed_utc', 'review_revision']],
        on='group_id', how='left', validate='one_to_one'
    )
    q['blind_decision'] = q.blind_decision.fillna('')
    def decode(r):
        if r.blind_decision == 'No visible difference': return 'No visible difference'
        if r.blind_decision == 'Mixed / uncertain effect': return 'Mixed / uncertain effect'
        if r.blind_decision == f'Image {r.forced_side} visibly better': return 'Beneficial forced preservation'
        other = 'B' if r.forced_side == 'A' else 'A'
        if r.blind_decision == f'Image {other} visibly better': return 'Forced preservation harmful'
        return ''
    q['decoded_effect'] = q.apply(decode, axis=1)
    q['beneficial'] = q.decoded_effect.eq('Beneficial forced preservation')
    q['harmful'] = q.decoded_effect.eq('Forced preservation harmful')
    return q


def cluster_bootstrap(decoded):
    required = {
        PAGE_COL,
        'beneficial',
        'in_utility_top50',
        'in_pprint_top50',
    }
    missing = required - set(decoded.columns)

    if missing:
        raise RuntimeError(
            f'Bootstrap input missing columns: {sorted(missing)}'
        )

    q = decoded.copy()
    q[PAGE_COL] = q[PAGE_COL].astype(str)

    q['utility_n'] = q['in_utility_top50'].astype(int)
    q['utility_beneficial'] = (
        q['beneficial'].astype(int)
        * q['in_utility_top50'].astype(int)
    )

    q['pprint_n'] = q['in_pprint_top50'].astype(int)
    q['pprint_beneficial'] = (
        q['beneficial'].astype(int)
        * q['in_pprint_top50'].astype(int)
    )

    page = (
        q.groupby(PAGE_COL, sort=True)
        .agg(
            utility_n=('utility_n', 'sum'),
            utility_beneficial=('utility_beneficial', 'sum'),
            pprint_n=('pprint_n', 'sum'),
            pprint_beneficial=('pprint_beneficial', 'sum'),
        )
        .reset_index()
    )

    if page.empty:
        raise RuntimeError('No page-level review rows available.')

    values = page[
        [
            'utility_n',
            'utility_beneficial',
            'pprint_n',
            'pprint_beneficial',
        ]
    ].to_numpy(dtype=np.float64)

    n_pages = len(values)
    rng = np.random.default_rng(BOOTSTRAP_SEED)

    output = []
    remaining = BOOTSTRAP_RESAMPLES
    resample_number = 0

    # Batching limits memory while avoiding repeated DataFrame copying.
    batch_size = 1000

    while remaining > 0:
        batch = min(batch_size, remaining)

        sampled_indices = rng.integers(
            0,
            n_pages,
            size=(batch, n_pages),
        )

        sampled = values[sampled_indices]
        totals = sampled.sum(axis=1)

        utility_denominator = totals[:, 0]
        utility_numerator = totals[:, 1]
        pprint_denominator = totals[:, 2]
        pprint_numerator = totals[:, 3]

        valid = (
            (utility_denominator > 0)
            & (pprint_denominator > 0)
        )

        utility_rate = (
            utility_numerator[valid]
            / utility_denominator[valid]
        )

        pprint_rate = (
            pprint_numerator[valid]
            / pprint_denominator[valid]
        )

        for ur, pr in zip(utility_rate, pprint_rate):
            output.append({
                'resample': resample_number,
                'utility_beneficial_rate': float(ur),
                'pprint_beneficial_rate': float(pr),
                'difference_utility_minus_pprint': float(ur - pr),
            })
            resample_number += 1

        remaining -= batch

    result = pd.DataFrame(output)

    if result.empty:
        raise RuntimeError(
            'No valid page-clustered bootstrap samples were produced.'
        )

    return result

final, message = finalise_analysis(require_complete=True)

print(message)
print(json.dumps(final, indent=2, default=str))


def finalise_analysis(require_complete=True):
    decoded = decoded_reviews()
    reviewed = decoded.blind_decision.astype(str).str.len().gt(0)
    if require_complete and not reviewed.all():
        remaining = int((~reviewed).sum())
        return None, f'Analysis remains blinded: {remaining} union groups are unreviewed.'
    q = decoded[reviewed].copy()
    if q.empty:
        return None, 'No completed reviews.'

    summary_rows = []
    for ranking, flag in [('reconstruction_avoided_utility', 'in_utility_top50'), ('mean_p_print', 'in_pprint_top50')]:
        s = q[q[flag]].copy()
        n = len(s); ben = int(s.beneficial.sum()); harm = int(s.harmful.sum())
        nodiff = int(s.decoded_effect.eq('No visible difference').sum())
        uncertain = int(s.decoded_effect.eq('Mixed / uncertain effect').sum())
        lo, hi = wilson_interval(ben, n)
        summary_rows.append({
            'ranking': ranking, 'reviewed_groups': n,
            'beneficial': ben, 'beneficial_rate': ben/max(n,1),
            'beneficial_wilson95_low': lo, 'beneficial_wilson95_high': hi,
            'harmful': harm, 'harmful_rate': harm/max(n,1),
            'no_visible_difference': nodiff, 'mixed_uncertain': uncertain,
            'pages': int(s[PAGE_COL].astype(str).nunique()),
        })
    summary = pd.DataFrame(summary_rows)

    # Page-clustered common-union bootstrap.
    bootstrap = cluster_bootstrap(q)
    if len(bootstrap):
        for ranking, col in [('reconstruction_avoided_utility', 'utility_beneficial_rate'), ('mean_p_print', 'pprint_beneficial_rate')]:
            summary.loc[summary.ranking.eq(ranking), 'beneficial_page_bootstrap95_low'] = float(bootstrap[col].quantile(.025))
            summary.loc[summary.ranking.eq(ranking), 'beneficial_page_bootstrap95_high'] = float(bootstrap[col].quantile(.975))
        diff_low = float(bootstrap.difference_utility_minus_pprint.quantile(.025))
        diff_high = float(bootstrap.difference_utility_minus_pprint.quantile(.975))
        prob_positive = float((bootstrap.difference_utility_minus_pprint > 0).mean())
    else:
        diff_low = diff_high = prob_positive = np.nan

    # Supplemental rank-position paired comparison.
    by_id = q.set_index('group_id')
    paired_rows = []
    for rank in range(1, TOP_K + 1):
        ug = utility_ids[rank-1]; pg = pprint_ids[rank-1]
        if ug not in by_id.index or pg not in by_id.index:
            continue
        ub = bool(by_id.loc[ug, 'beneficial']); pb = bool(by_id.loc[pg, 'beneficial'])
        paired_rows.append({'rank_position': rank, 'utility_group_id': ug, 'pprint_group_id': pg, 'utility_beneficial': ub, 'pprint_beneficial': pb})
    paired = pd.DataFrame(paired_rows)
    u_only = int(((paired.utility_beneficial) & (~paired.pprint_beneficial)).sum()) if len(paired) else 0
    p_only = int(((~paired.utility_beneficial) & (paired.pprint_beneficial)).sum()) if len(paired) else 0
    discordant = u_only + p_only
    sign_p = float(binomtest(u_only, discordant, .5, alternative='two-sided').pvalue) if discordant else 1.0
    paired['utility_only_wins_total'] = u_only
    paired['pprint_only_wins_total'] = p_only
    paired['discordant_total'] = discordant
    paired['exact_two_sided_sign_test_p'] = sign_p

    decoded_out = q.copy()
    decoded_out.to_csv(REVIEW / 'decoded_blinded_group_review.csv', index=False)
    summary.to_csv(SUMMARY_CSV, index=False)
    paired.to_csv(PAIRED_CSV, index=False)
    bootstrap.to_csv(BOOTSTRAP_CSV, index=False)
    with pd.ExcelWriter(SUMMARY_XLSX, engine='openpyxl') as writer:
        summary.to_excel(writer, sheet_name='beneficial_rates', index=False)
        paired.to_excel(writer, sheet_name='rank_position_pairing', index=False)
        decoded_out.to_excel(writer, sheet_name='decoded_reviews', index=False)

    final = {
        'completed_utc': datetime.now(timezone.utc).isoformat(),
        'union_reviewed': int(len(q)),
        'utility_top_k': TOP_K,
        'pprint_top_k': TOP_K,
        'overlap': int(len(set(utility_ids) & set(pprint_ids))),
        'rate_summary': summary.to_dict('records'),
        'page_cluster_bootstrap_difference_utility_minus_pprint_95': [diff_low, diff_high],
        'page_cluster_bootstrap_probability_difference_positive': prob_positive,
        'rank_position_paired': {
            'utility_only_beneficial': u_only,
            'pprint_only_beneficial': p_only,
            'discordant_positions': discordant,
            'exact_two_sided_sign_test_p': sign_p,
            'interpretation_note': 'Supplemental rank-position pairing; primary uncertainty is the page-clustered common-union bootstrap.',
        },
    }
    FINAL_JSON.write_text(json.dumps(final, indent=2, default=str))
    message = (
        f'Final analysis written. Utility beneficial rate: '
        f"{summary.loc[summary.ranking.eq('reconstruction_avoided_utility'),'beneficial_rate'].iloc[0]:.1%}; "
        f"p_print beneficial rate: {summary.loc[summary.ranking.eq('mean_p_print'),'beneficial_rate'].iloc[0]:.1%}."
    )
    return final, message

persist_review()
print('Review rows:', len(review_df))

## 8. Blinded Gradio reviewer

The UI deliberately does not reveal ranking source, utility, `p_print`, clean truth, difference maps, page ID, group ID or the baseline/forced side assignment. Results remain locked until all union groups have a saved judgement.

In [ ]:
def filtered_indices(filter_name):
    if filter_name == 'Unreviewed only':
        return review_df.index[review_df.blind_decision.astype(str).str.len().eq(0)].tolist()
    if filter_name == 'Reviewed only':
        return review_df.index[review_df.blind_decision.astype(str).str.len().gt(0)].tolist()
    return review_df.index.tolist()


def progress_text():
    done = int(review_df.blind_decision.astype(str).str.len().gt(0).sum())
    return f'**Reviewed: {done} / {len(review_df)}** — analysis remains blinded until all records are complete.'


def render(filter_name, pos):
    ids = filtered_indices(filter_name)
    if not ids:
        blank = np.full((420, 620, 3), 245, np.uint8)
        return blank, blank, 'No matching records', '', '', 0, progress_text()
    pos = max(0, min(int(pos), len(ids)-1))
    idx = ids[pos]
    r = review_df.loc[idx]
    a, b = render_group_pair(r.group_id)
    meta = f'**Blinded record {r.review_id} — {pos+1}/{len(ids)} in current filter**'
    return a, b, meta, str(r.blind_decision), str(r.review_note), pos, progress_text()


def move(filter_name, pos, delta):
    ids = filtered_indices(filter_name)
    return render(filter_name, 0 if not ids else (int(pos)+delta) % len(ids))


def save_review(filter_name, pos, decision, note):
    ids = filtered_indices(filter_name)
    if not ids or decision not in DECISIONS:
        return (*render(filter_name, pos), 'Select one of the four review decisions before saving.')
    idx = ids[max(0, min(int(pos), len(ids)-1))]
    with LOCK:
        review_df.at[idx, 'blind_decision'] = decision
        review_df.at[idx, 'review_note'] = str(note or '')
        review_df.at[idx, 'reviewer'] = REVIEWER_NAME
        review_df.at[idx, 'reviewed_utc'] = datetime.now(timezone.utc).isoformat()
        review_df.at[idx, 'review_revision'] = int(review_df.at[idx, 'review_revision']) + 1
        persist_review()
        complete = review_df.blind_decision.astype(str).str.len().gt(0).all()
        if complete:
            _, status = finalise_analysis(require_complete=True)
        else:
            status = 'Saved. Final ranking analysis remains locked until all union groups are reviewed.'
    # Under the unreviewed filter, the saved row disappears; keep the same numeric position.
    next_pos = int(pos) if filter_name == 'Unreviewed only' else int(pos) + 1
    return (*render(filter_name, next_pos), status)


def finalise_button():
    _, message = finalise_analysis(require_complete=True)
    files = [str(REVIEW_CSV), str(REVIEW_XLSX)]
    for p in [SUMMARY_CSV, SUMMARY_XLSX, PAIRED_CSV, BOOTSTRAP_CSV, FINAL_JSON]:
        if p.exists(): files.append(str(p))
    return message, files


def export_current():
    persist_review()
    files = [str(REVIEW_CSV), str(REVIEW_XLSX), str(TABLES/'blinded_union_sampling_manifest.csv')]
    for p in [SUMMARY_CSV, SUMMARY_XLSX, PAIRED_CSV, BOOTSTRAP_CSV, FINAL_JSON]:
        if p.exists(): files.append(str(p))
    return files


with gr.Blocks(title='Notebook 36 — Blinded Utility vs p_print Group Review') as demo:
    gr.Markdown('# Notebook 36 — Blinded Utility-versus-p_print Group Review')
    gr.Markdown(
        'Judge only which rendered output looks better inside the hollow purple box. '
        '**Image A/B assignment is random and fixed. Ranking source, probability, utility, clean truth and difference maps are hidden.**'
    )
    progress = gr.Markdown(progress_text())
    with gr.Row():
        filt = gr.Dropdown(['Unreviewed only', 'Reviewed only', 'All'], value='Unreviewed only', label='Filter')
        refresh = gr.Button('↻ Refresh')
        finalise = gr.Button('Finalise analysis', variant='secondary')
    with gr.Row():
        image_a = gr.Image(interactive=False, label='Image A', height=560)
        image_b = gr.Image(interactive=False, label='Image B', height=560)
    meta = gr.Markdown()
    decision = gr.Radio(DECISIONS, label='Blinded visual judgement')
    note = gr.Textbox(label='Optional note', lines=2)
    with gr.Row():
        prev = gr.Button('◀ Previous')
        save_btn = gr.Button('Save & Next', variant='primary')
        nxt = gr.Button('Next ▶')
        export_btn = gr.Button('Export current record')
    status = gr.Markdown()
    files = gr.Files(label='Exports')
    pos = gr.State(0)
    outs = [image_a, image_b, meta, decision, note, pos, progress]

    demo.load(lambda: render('Unreviewed only', 0), outputs=outs)
    filt.change(lambda f: render(f, 0), inputs=filt, outputs=outs)
    refresh.click(render, inputs=[filt, pos], outputs=outs)
    prev.click(lambda f, p: move(f, p, -1), inputs=[filt, pos], outputs=outs)
    nxt.click(lambda f, p: move(f, p, 1), inputs=[filt, pos], outputs=outs)
    save_btn.click(save_review, inputs=[filt, pos, decision, note], outputs=outs + [status])
    finalise.click(finalise_button, outputs=[status, files])
    export_btn.click(export_current, outputs=files)

print('Blinded reviewer ready. Union groups:', len(review_df))

## 9. Experiment identity

In [ ]:
environment = {
    'python': sys.version,
    'opencv': cv2.__version__,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'gradio': gr.__version__,
    'platform': platform.platform(),
}
identity = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'version': '8.2g-blinded-utility-versus-pprint-group-review',
    'development_only': True,
    'promotion_authority': False,
    'retrained': False,
    'refitted': False,
    'recalibrated': False,
    'source_notebook35_run': str(V82F),
    'source_notebook35_identity_sha256': sha256_file(V82F/'experiment_identity.json'),
    'source_group_table': str(GROUP_PATH),
    'source_group_table_sha256': sha256_file(GROUP_PATH),
    'source_membership_file': str(MEMBERSHIP_SOURCE),
    'source_membership_sha256': sha256_file(MEMBERSHIP_SOURCE),
    'source_v82_run': str(V82),
    'top_k_each_ranking': TOP_K,
    'selection_summary': selection_summary,
    'review_seed': REVIEW_SEED,
    'bootstrap_seed': BOOTSTRAP_SEED,
    'bootstrap_resamples': BOOTSTRAP_RESAMPLES,
    'review_decisions': DECISIONS,
    'blinding': {
        'ranking_origin_hidden': True,
        'metrics_hidden': True,
        'clean_hidden': True,
        'difference_hidden': True,
        'baseline_forced_side_randomised': True,
        'results_locked_until_complete': True,
    },
    'software': environment,
}
(OUT/'experiment_identity.json').write_text(json.dumps(identity, indent=2, default=str))
(OUT/'software_environment.json').write_text(json.dumps(environment, indent=2))
print(json.dumps(identity, indent=2, default=str))

## 10. Launch reviewer

In [ ]:
demo.queue(default_concurrency_limit=1).launch(share=PUBLIC_LINK, debug=True)

## Outputs to return

After all groups are reviewed, return:

- `final_review_analysis.json`
- `tables/ranking_beneficial_rate_summary.csv`
- `tables/paired_rank_position_comparison.csv`
- `tables/page_cluster_bootstrap_rate_difference.csv`
- `review/decoded_blinded_group_review.csv`
- `review/blinded_group_review.csv`
- `experiment_identity.json`

The primary comparison is the beneficial-rate difference and its page-clustered bootstrap interval. The Wilson intervals are descriptive group-level intervals. The rank-position sign test is supplemental.